# 4.2h — Bench `yolov5nu` sur le terrain du 4.2c (chunk 2/3, EPF #16057)

[← 4.2g — YOLO ultralytics (yolo11n/s)](4.2g-Detection-SOTA-Ultralytics.ipynb) | [4.2c — AnchorNet from scratch](4.2c-Detection-Anchor-From-Scratch.ipynb)

Le [4.2g](4.2g-Detection-SOTA-Ultralytics.ipynb) a livré `yolo11n` et `yolo11s` fine-tunés sur notre terrain synthétique. Ce notebook ajoute **la troisieme reference de la famille YOLO** : `yolov5nu` — la version `u` (unifiée Ultralytics 8.4) du modele YOLOv5. Meme terrain, meme budget (1 000 images x 6 époques), meme `ap_voc` maison — pour que les trois lignes se lisent sur un meme tableau.

Ce chunk 2/3 de l'EPF #16057 ferme la promesse du #16337 : aux cotes de `yolo11n` et `yolo11s`, le terrain a ete mesure pour `yolov5nu`. Le `yolov8s` (autre membre de la fratrie) reste a livrer en chunk 3/3.

**Note d'honnetete** : `yolov5nu.pt` (suffixe `u`) est la version Ultralytics unifiee, pas le YOLOv5 2020 d'origine. On compare ici des modeles Ultralytics 8.4.0 entre eux, pas l'histoire de la detection en mouvement.

In [1]:
import json
import time
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import cv2
from ultralytics import YOLO

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
import ultralytics
print("device:", "cuda" if DEVICE == 0 else "cpu",
      "| torch", torch.__version__, "| ultralytics", ultralytics.__version__)

IMG = 96          # terrain meme cote qu'en 4.2c/4.2f/4.2g (multiple de 32)

device: cuda | torch 2.6.0+cu124 | ultralytics 8.4.153


**Lecture.** Le banc tourne sur GPU (`cuda`, torch 2.6.0+cu124) avec
ultralytics **8.4.153** — la version qui porte les poids `u` unifiés. Ce trio
device/torch/ultralytics est commité en tête de notebook parce que tout le
tableau final en dépend : les latences et les temps d'entraînement ne se
comparent qu'à environnement constant. Le `IMG = 96` n'est pas arbitraire :
les réseaux YOLO divisent par 2 à chaque étage de la colonne (stride 32 au
plus profond), et 96 est multiple de 32 — la seule contrainte matérielle
que le terrain doive au modèle.

## 1. Terrain : générateur verbatim du 4.2c

Memes graines, meme recettes de blobs + boites — 2 000 images d'entrainement, 400 images / 817 GT de validation. **L'identite du terrain est ce qui rend la comparaison avec les mesures committées du 4.2g valide**.L'identité n'est pas un mot : mêmes graines → mêmes images → mêmes boîtes GT
que les runs committés du 4.2g. Toute dérive (un paramètre du générateur, une
autre graine) invaliderait les deux lignes 4.2g du tableau final — c'est pour
ça que le vérificateur le plus simple est déjà écrit : les comptes imprimés
ci-dessous (2000 / 400 / 817) doivent reproduire ceux du 4.2g à l'identique.

In [2]:
def iou_np(a, b):
    """IoU scalaire (x0, y0, w, h) numpy."""
    ix = max(0.0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / union if union > 0 else 0.0


def make_image(rng):
    img = rng.normal(0, 0.08, (IMG, IMG)).astype(np.float32)
    yy, xx = np.mgrid[0:IMG, 0:IMG]
    for _ in range(rng.integers(2, 5)):
        cy, cx = rng.integers(0, IMG, 2)
        s = rng.uniform(18, 50)
        img += 0.10 * rng.uniform(0.6, 1.3) * np.exp(-(((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * s * s)))
    boxes = []
    for _ in range(rng.integers(1, 4)):
        kind = rng.choice(["rect", "ellipse"])
        for _try in range(30):
            w = int(rng.uniform(14, 44))
            h = int(max(10, min(48, w * rng.uniform(0.35, 2.9))))
            x0 = int(rng.integers(2, IMG - w - 2))
            y0 = int(rng.integers(2, IMG - h - 2))
            cand = (x0, y0, w, h)
            if all(iou_np(cand, b) < 0.25 for b in boxes):
                boxes.append(cand)
                break
    for (x0, y0, w, h) in boxes:
        amp = rng.uniform(0.7, 1.2)
        if kind == "rect":
            img[y0:y0 + h, x0:x0 + w] += amp
        else:
            sub = img[y0:y0 + h, x0:x0 + w]
            ey, ex = np.mgrid[0:h, 0:w]
            mask = (((ex - w / 2) / (w / 2)) ** 2 + ((ey - h / 2) / (h / 2)) ** 2) <= 1.0
            img[y0:y0 + h, x0:x0 + w] = np.where(mask, sub + amp, sub)
    return np.clip(img, -1.5, 2.5), boxes


def make_split(n, seed):
    rng = np.random.default_rng(seed)
    xs, bs = [], []
    for _ in range(n):
        img, boxes = make_image(rng)
        xs.append(img)
        bs.append(torch.tensor(boxes, dtype=torch.float32))
    return torch.tensor(np.stack(xs)).unsqueeze(1), bs


Xtr, Btr = make_split(2000, SEED + 1)
Xva, Bva = make_split(400, SEED + 2)
print("train:", tuple(Xtr.shape), "| val:", tuple(Xva.shape),
      "| objets GT val:", sum(len(b) for b in Bva))

train: (2000, 1, 96, 96) | val: (400, 1, 96, 96) | objets GT val: 817


**Lecture.** Le terrain est byte-identique à celui du 4.2c/4.2g parce que le
générateur ET les graines sont repris verbatim (`SEED+1` train, `SEED+2`
val) : 2 000 images d'entraînement, 400 de validation portant **817 objets
GT** — soit en moyenne ~2 objets par image, avec 1 à 3 formes (`rect` ou
`ellipse`) posées sur un fond de 2 à 4 taches gaussiennes. Le rejet
`IoU < 0.25` entre boites garantit des objets non chevauchants : chaque
détection a une cible non ambiguë, ce qui rend le matching glouton de la
section 4 (IoU ≥ 0,5, par score décroissant) une procédure sûre. C'est
cette identité du terrain qui autorise le tableau final à mettre en regard
les valeurs committées du 4.2g : trois modèles, un seul terrain.

## 2. Dataset au format YOLO

Meme conversion uint8 + labels normalises + `data.yaml` que dans le 4.2g — aucun raccourci.

In [3]:
DS = Path(tempfile.mkdtemp(prefix="terrain_yolov5_"))


def write_yolo_split(X, B, split):
    (DS / "images" / split).mkdir(parents=True, exist_ok=True)
    (DS / "labels" / split).mkdir(parents=True, exist_ok=True)
    for i, (img, boxes) in enumerate(zip(X, B)):
        u8 = (((img - img.min()) / (img.max() - img.min() + 1e-6)) * 255).astype(np.uint8)
        cv2.imwrite(str(DS / "images" / split / f"{i:05d}.png"), u8)
        lines = [f"0 {(x0 + w / 2) / IMG} {(y0 + h / 2) / IMG} {w / IMG} {h / IMG}"
                 for (x0, y0, w, h) in boxes]
        (DS / "labels" / split / f"{i:05d}.txt").write_text(
            "\n".join(lines), encoding="utf-8")


write_yolo_split([x[0].numpy() for x in Xtr], [b.tolist() for b in Btr], "train")
write_yolo_split([x[0].numpy() for x in Xva], [b.tolist() for b in Bva], "val")
(DS / "data.yaml").write_text(
    f"path: {DS.as_posix()}\ntrain: images/train\nval: images/val\nnc: 1\nnames: ['objet']\n",
    encoding="utf-8")
print("dataset YOLO pret :", len(list((DS / "images" / "train").glob("*.png"))), "train /",
      len(list((DS / "images" / "val").glob("*.png"))), "val, 1 classe")

dataset YOLO pret : 2000 train / 400 val, 1 classe


**Lecture.** La conversion applique à chaque image un min-max **par image**
avant le cast `uint8` : une légère compression de dynamique (le fond
gaussien ±0,08 devient 0-255), identique sur les trois notebooks — le biais
est commun, donc la comparaison reste loyale. Les labels YOLO sont
normalisés en `xywh` relatif (`w/IMG`, `h/IMG`) : le format qu'attend la
perte du détecteur. `data.yaml` déclare `nc: 1` — la classe unique `objet`.

## 3. Fine-tuning `yolov5nu`

Budget identique au 4.2g : 1 000 images x 6 époques, `imgsz=96`, `batch=16`. Le pré-entrainement COCO telecharge le modele au premier appel (~5 Mo).Deux choix méritent d'être nommés : `fraction = 1000/2000` laisse Ultralytics
échantillonner la moitié du dataset sous graine fixée (contrôle du budget sans
découpage manuel, la sélection est reproductible) ; `workers=0` et
`verbose=False` rendent le run hermétique — aucune dépendance au parallélisme
de chargement ni aux barres de progression dans les sorties committées. Le
pré-entraînement COCO (~5 Mo) n'est téléchargé qu'au premier appel du
`YOLO("yolov5nu.pt")`.

In [4]:
N_TRAIN, EPOCHS = 1000, 6
print("budget commun 4.2f / 4.2g / 4.2h :", N_TRAIN, "images x", EPOCHS, "époques")

from ultralytics.utils import LOGGER
LOGGER.setLevel("WARNING")


def train_yolo(weights, run):
    m = YOLO(weights)
    torch.manual_seed(SEED)
    t0 = time.time()
    m.train(data=str(DS / "data.yaml"), epochs=EPOCHS, imgsz=IMG,
            batch=16, device=DEVICE, fraction=N_TRAIN / len(Xtr),
            project=str(DS / "runs"), name=run, exist_ok=True,
            verbose=False, plots=False, seed=SEED, workers=0)
    return m, time.time() - t0


MODEL_5N, T_5N = train_yolo("yolov5nu.pt", "yolov5nu")
_, params_5n, _, flops_5n = MODEL_5N.info(imgsz=IMG)
print(f"\nyolov5nu fine-tune termine en {T_5N:.1f} s, params={params_5n:,}, GFLOPs={flops_5n:.1f}")

budget commun 4.2f / 4.2g / 4.2h : 1000 images x 6 époques


WARNING train: Slow image access detected (ping: 0.10.1 ms, read: 3.82.0 MB/s, size: 6.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips


WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 2.21.8 MB/s, size: 7.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips



yolov5nu fine-tune termine en 117.5 s, params=2,508,659, GFLOPs=0.2


**Lecture.** Le fine-tune tient en **117,5 s** pour le budget commun
(1 000 images × 6 époques) : plus du double du temps des jumeaux `yolo11`
(57 s / 54 s dans le 4.2g). Le modèle livré porte **2 508 659 paramètres** —
3 % de moins que `yolo11n` (2 590 035), l'écart le plus serré de la fratrie
— mais **0,2 GFLOPs** au dixième imprimé contre 0,1 : à taille quasi égale,
l'architecture yolov5-u demande le double de calcul par image. Les deux
`WARNING slow image access` sont cosmétiques : ping 0,1 ms et images de
6,9 Ko dans un répertoire temporaire — le conseil de stockage local vise
les datasets distants, pas ce banc. Graine fixée (`seed=SEED`, `workers=0`)
: le run est reproductible.

## 4. Évaluation : `ap_voc` maison, seuils du 4.2c

`model.predict(conf=0.5, iou=0.45)`, matching glouton IoU ≥ 0,5, `ap_voc` repris verbatim du 4.2g — protocole strictement identique pour comparer aux deux mesures committées dans ce notebook.Pourquoi une métrique maison plutôt que les `model.val()` d'Ultralytics :
comparabilité. Le protocole du 4.2c (matching glouton par score décroissant,
IoU ≥ 0,5, `conf=0.5`, `iou=0.45`) est celui qui a produit les valeurs
committées du 4.2f et du 4.2g — changer d'instrument entre deux lignes d'un
même tableau serait changer la question en cours de réponse.

In [5]:
def iou_t(boxes1, boxes2):
    """IoU vectorisee (N,4) x (M,4) en (x0,y0,w,h) -> (N,M)."""
    b1, b2 = boxes1.to(DEVICE), boxes2.to(DEVICE)
    ix0 = torch.maximum(b1[:, None, 0], b2[None, :, 0])
    iy0 = torch.maximum(b1[:, None, 1], b2[None, :, 1])
    ix1 = torch.minimum(b1[:, None, 0] + b1[:, None, 2], b2[None, :, 0] + b2[None, :, 2])
    iy1 = torch.minimum(b1[:, None, 1] + b1[:, None, 3], b2[None, :, 1] + b2[None, :, 3])
    iw = (ix1 - ix0).clamp(min=0)
    ih = (iy1 - iy0).clamp(min=0)
    inter = iw * ih
    union = b1[:, None, 2] * b1[:, None, 3] + b2[None, :, 2] * b2[None, :, 3] - inter
    return inter / (union + 1e-9)


def predict_boxes(model, idx):
    """Boites (N,4) xyxy + scores d'une image, seuils du 4.2c."""
    r = model.predict(str(DS / "images" / "val" / f"{idx:05d}.png"),
                      conf=0.5, iou=0.45, imgsz=IMG, verbose=False,
                      device=DEVICE)[0].boxes
    return (torch.tensor(r.xyxy.tolist(), dtype=torch.float32),
            torch.tensor(r.conf.tolist(), dtype=torch.float32))


def xyxy_yolo(b_xywh):
    t = torch.tensor(b_xywh, dtype=torch.float32)
    return torch.stack([t[:, 0], t[:, 1], t[:, 0] + t[:, 2], t[:, 1] + t[:, 3]], dim=1)


def collect_pr_yolo(model, B, iou_thr=0.5):
    tp_s, fp_s, ngt = [], [], 0
    for i in range(len(B)):
        dets, dscores = predict_boxes(model, i)
        gts = xyxy_yolo(B[i])
        matched = torch.zeros(len(gts), dtype=torch.bool)
        for d in dscores.argsort(descending=True).tolist():
            if len(gts):
                ious = iou_t(dets[d:d + 1], gts)[0]
                ious[matched] = -1
                g = int(ious.argmax())
                if ious[g] >= iou_thr:
                    matched[g] = True
                    tp_s.append(float(dscores[d]))
                    continue
            fp_s.append(float(dscores[d]))
        ngt += len(B[i])
    return tp_s, fp_s, ngt


def ap_voc(tp_s, fp_s, ngt):
    flags = np.array([1] * len(tp_s) + [0] * len(fp_s), dtype=np.float64)
    scores = np.array(tp_s + fp_s, dtype=np.float64)
    order = np.argsort(-scores)
    flags, scores = flags[order], scores[order]
    ctp, cfp = np.cumsum(flags), np.cumsum(1 - flags)
    rec = ctp / max(ngt, 1)
    prec = ctp / np.maximum(ctp + cfp, 1e-9)
    mrec = np.concatenate([[0], rec, [1]])
    mpre = np.concatenate([[0], prec, [0]])
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    ap10 = float(np.sum((mrec[1:] - mrec[:-1]) * mpre[1:]))
    ap07 = 0.0
    for t in np.linspace(0, 1, 11):
        sel = rec >= t
        ap07 += (prec[sel].max() if sel.any() else 0.0) / 11
    return ap07, ap10, rec, prec


tp_s, fp_s, ngt = collect_pr_yolo(MODEL_5N, [b.tolist() for b in Bva])
ap07, ap10, rec, prec = ap_voc(tp_s, fp_s, ngt)
print(f"yolov5nu  mAP@0.5 sur 400 images / {ngt} objets GT : "
      f"VOC07 11-point {ap07:.3f} | VOC10 all-point {ap10:.3f}")

yolov5nu  mAP@0.5 sur 400 images / 817 objets GT : VOC07 11-point 0.909 | VOC10 all-point 0.972


**Lecture.** Sur 400 images et 817 objets GT : **VOC07 11-point 0,909 ·
VOC10 all-point 0,972**. L'écart de 6,3 points entre les deux métriques dit
où vit la difficulté du terrain : l'`all-point` intègre toute la courbe
précision/rappel (il récompense la queue de rappel profonde, là où les
derniers objets sont retrouvés à basse confiance), tandis que le 11-point
échantillonne la précision à 11 seuils de rappel. Le fait saillant se lit
en croisant avec le 4.2g : **VOC07 = 0,909 pour les trois modèles**
(yolo11n, yolo11s, yolov5nu) — à ce budget, la métrique 11-point ne sépare
personne ; toute la discrimination vit dans VOC10, où yolov5nu rend
1,7 point à yolo11n (0,972 vs 0,989) et 2,1 points à yolo11s (vs 0,993).

## 5. Latence d'inference (GPU) et tableau comparatif chunk 2/3

Memes 100 images, meme synchronisation CUDA. Le tableau met en regard `yolov5nu` (mesure c.1217) avec les valeurs committées du 4.2g (`yolo11n`, `yolo11s`). Les trois membres de la famille YOLO sur un terrain et un budget identiques.La discipline de mesure est asynchrone-aware : 5 prédictions d'échauffement
(compile/JIT amortis), puis `torch.cuda.synchronize()` avant ET après la
boucle — sans elle, le chronomètre mesurerait le temps de *lancement* des
noyaux CUDA, pas leur exécution. Les 100 images sont moyennées.

In [6]:
def latency_ms_yolo(model, n=100):
    for i in range(5):
        predict_boxes(model, i)
    if DEVICE == 0:
        torch.cuda.synchronize()
    t0 = time.time()
    for i in range(n):
        predict_boxes(model, i)
    if DEVICE == 0:
        torch.cuda.synchronize()
    return (time.time() - t0) * 1000.0 / n


lat_5n = latency_ms_yolo(MODEL_5N)

# valeurs committées dans le 4.2g -- source: tableau bloc B l.723-724
ROWS_45H = [
    ("4.2g yolo11n",  2_590_035, 0.1, 0.909, 0.989, 10.3, f"{N_TRAIN} x {EPOCHS} en 57 s"),
    ("4.2g yolo11s",  9_428_179, 0.5, 0.909, 0.993, 10.8, f"{N_TRAIN} x {EPOCHS} en 54 s"),
    ("4.2h yolov5nu", params_5n, flops_5n, ap07, ap10, lat_5n, f"{N_TRAIN} x {EPOCHS} en {T_5N:.0f} s"),
]
print(f"\n{'modele':12s} {'params':>11s} {'GFLOPs':>7s} {'VOC07':>6s} {'VOC10':>6s} {'ms/img':>7s}  budget")
for r in ROWS_45H:
    print(f"{r[0]:12s} {r[1]:>11,} {r[2]:7.1f} {r[3]:6.3f} {r[4]:6.3f} {r[5]:7.1f}  {r[6]}")


modele            params  GFLOPs  VOC07  VOC10  ms/img  budget
4.2g yolo11n   2,590,035     0.1  0.909  0.989    10.3  1000 x 6 en 57 s
4.2g yolo11s   9,428,179     0.5  0.909  0.993    10.8  1000 x 6 en 54 s
4.2h yolov5nu   2,508,659     0.2  0.909  0.972    18.4  1000 x 6 en 118 s


**Lecture.** Le tableau se lit colonne par colonne, et chaque colonne raconte
un fait distinct. **Params** : yolov5nu (2 508 659) ≈ yolo11n (2 590 035),
−3 % — l'unification `u` a bien ramené la fratrie 2020 sur le terrain
d'optimisation des yolo11 ; yolo11s reste 3,8× plus lourd (9 428 179).
**GFLOPs** : 0,2 contre 0,1 au dixième imprimé — le double de calcul par
image à taille égale. **Justesse** : VOC07 saturé (0,909 partout), et
c'est VOC10 qui ordonne : yolo11s (0,993) > yolo11n (0,989) > yolov5nu
(0,972) — la génération récente tient mieux la queue de rappel.
**Latence** : 18,4 ms/image contre 10,3 / 10,8 — le rapport ~1,8× est
cohérent avec le double de FLOPs. **Budget d'entraînement** : 118 s contre
57 / 54 — le coût d'apprentissage suit la même pente. Verdict de banc : à
paramètres égaux, yolov5nu paie 2× calcul et latence pour une queue de
rappel légèrement inférieure — sur ce terrain, `yolo11n` domine son aîné
unifié, et la comparaison est loyale car terrain, budget et protocole
d'évaluation sont identiques par construction.

## 6. Conclusion chunk 2/3 — yolov5nu face aux yolo11

Trois observations directes à partir du tableau :

- Le **prix d'un yolov5nu** tient dans les memes ordres de grandeur qu'un `yolo11n` (~2,6 M) — l'unification Ultralytics (`u`) a effectivement rapproche les deux fratries du même terrain d'optimisation. Les GFLOPs et la latence d'inference se lisent dans la colonne du milieu.
- La **justesse en detection** (mAP VOC07/VOC10) est ce qui bouge : sur notre terrain, la difference d'accuracy reflete les choix architecturaux (PAN-FPN, tete decouplée, anchor-free, etc.) plus que la famille marketing du modele. Le 4.2h documente la mesure ; le pourquoi est dans la litterature Ultralytics citée en fin du 4.2g.
- Le **yolov8s** manque encore pour fermer le comparatif fratrie-complete. Cette livraison est programmee en **chunk 3/3** (issue #16346 — meme ticket).

## 7. Limites et suite

- **Un seul membre de la fratrie** : yolov8s manque. La promesse 3-modeles YOLOv5/v8/v11 sera atteinte avec le chunk 3/3 sur la meme issue #16346.
- **`yolov5nu` n'est pas YOLOv5 2020** : c'est la version unifiee Ultralytics 8.4.0 — la comparaison est entre modeles Ultralytics 8.4 contemporains, pas entre generations de 2020 et 2024. L'aspect historique (« YOLOv5 -> YOLOv8 -> YOLOv11 ») reste l'objet du recit markdown prevu dans l'issue parente #16337, qui sera livre en parallele.
- **Memes limites que le 4.2g** : mono-classe, imgsz=96 fixe, terrain synthetique. Ces hypotheses sont documentees dans le 4.2g section §8 et restent vrais ici.